# 05 · Visualization

**Goal: make time series that tell the truth, including about gaps.**

Sensor data is messy: noisy at the level of individual samples, and full of holes
where nodes went offline. A plot that hides those holes is a lie by omission. This
notebook builds a plot that shows the raw scatter, a smoothed trend, and honest
breaks where data is missing, then compares two nodes on one axis. It runs offline
using the same generator as notebook `04`.

In [ ]:
# Offline-safe synthetic data generator.
#
# This produces records in the exact schema that sage_data_client.query returns,
# so every SageTools helper (analyzeFile, plotTemperature, auditFile) works on it
# unchanged. Use it when the classroom has no route to the Sage Data API. When
# you do have connectivity, replace the synthetic frame with a real query from
# notebook 02 or 03 and the rest of the notebook is identical.
#
# Nothing here is hidden or disguised: the function is named for what it is, the
# timestamps are fixed to mid 2024, and the values are generated, not measured.

import numpy as np
import pandas as pd


def makeSyntheticBeehive(vsn="W07C", days=14, stepMinutes=5, seed=7,
                         sensors=("bme680", "bme280")):
    """
    Build a Beehive-schema temperature DataFrame with a realistic diurnal
    signal, seasonal drift, per-sensor offsets, and a couple of outage gaps.

    Args:
        vsn: node call sign to stamp on every row.
        days: length of the record in days.
        stepMinutes: spacing between readings, in minutes.
        seed: random seed, so runs are reproducible.
        sensors: one or more sensor names to emit, each with a small bias.

    Returns:
        A DataFrame with columns timestamp, name, value, meta.vsn, meta.sensor,
        meta.host. Values are in Celsius, matching the real feed.
    """
    rng = np.random.default_rng(seed)
    periods = int(days * 24 * 60 / stepMinutes)
    times = pd.date_range("2024-06-01", periods=periods, freq=f"{stepMinutes}min", tz="UTC")

    hourOfDay = times.hour + times.minute / 60.0
    diurnal = 8.0 * np.sin((hourOfDay - 9.0) / 24.0 * 2 * np.pi)  # afternoon peak
    dayIndex = (times - times[0]).days
    seasonalDrift = 0.15 * dayIndex                               # slow warming
    baseline = 21.0

    frames = []
    for offsetIdx, sensor in enumerate(sensors):
        sensorBias = 0.6 * offsetIdx                             # each sensor reads slightly high
        noise = rng.normal(0, 0.4, size=periods)
        celsius = baseline + diurnal + seasonalDrift + sensorBias + noise
        frame = pd.DataFrame({
            "timestamp": times,
            "name": "env.temperature",
            "value": celsius,
            "meta.vsn": vsn,
            "meta.sensor": sensor,
            "meta.host": f"000048b02d{15 + offsetIdx}a.ws-nxcore",
        })
        frames.append(frame)

    combined = pd.concat(frames, ignore_index=True).sort_values("timestamp").reset_index(drop=True)

    # Punch two realistic outage gaps into the record.
    for gapStart, gapHours in [(3, 6), (9, 14)]:
        lo = times[0] + pd.Timedelta(days=gapStart)
        hi = lo + pd.Timedelta(hours=gapHours)
        combined = combined[~combined["timestamp"].between(lo, hi)].reset_index(drop=True)

    return combined

In [ ]:
# Import the SageTools helper package (github.com/mpapka/SageTools).
#
# SageTools is a convenience layer built on top of the official sage_data_client.
# It is optional. Every lab in this course also shows the underlying
# sage_data_client call directly, so the helper is never a hard dependency.
#
# The try/except means the notebook runs whether or not the package is installed:
# if it is missing, haveSageTools becomes False and the notebook falls back to
# the direct calls it already teaches. To install it, see notebook 00.
try:
    import sage
    haveSageTools = True
except ImportError:
    haveSageTools = False

print("SageTools helper package available:", haveSageTools)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

# A sensible default figure size for time series.
plt.rcParams["figure.figsize"] = (12, 6)

In [ ]:
# Build a single-sensor record to plot, and add a Fahrenheit column.
plotDf = makeSyntheticBeehive(vsn="W06C", days=14, sensors=("bme680",))
plotDf["timestamp"] = pd.to_datetime(plotDf["timestamp"])
plotDf["valueF"] = plotDf["value"] * 9 / 5 + 32
plotDf = plotDf.sort_values("timestamp").reset_index(drop=True)
plotDf.head()

## 1. A first plot

Start with the simplest thing that could work: temperature against time. Even a
bare line reveals the daily rhythm and the slow warming trend. The date formatter
keeps the x axis readable.

In [ ]:
plt.plot(plotDf["timestamp"], plotDf["valueF"], color="tab:blue", linewidth=0.6)
plt.xlabel("Time")
plt.ylabel("Temperature (F)")
plt.title("W06C temperature, raw")
plt.grid(True, linestyle="--", alpha=0.5)
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 2. Add a rolling average

The raw line is noisy. A rolling average over a one hour window smooths the
sample noise while keeping the daily shape. The key move is setting `timestamp` as
the index so pandas can use a time based window (`"1h"`) rather than a fixed
number of rows, which matters when the sampling interval is not perfectly even.

In [ ]:
indexed = plotDf.set_index("timestamp")
rollingAvg = indexed["valueF"].rolling(window="1h", min_periods=1).mean()

plt.plot(plotDf["timestamp"], plotDf["valueF"], ".", color="tab:blue",
         alpha=0.15, markersize=2, label="raw samples")
plt.plot(rollingAvg.index, rollingAvg.values, color="tab:red",
         linewidth=1.5, label="1 hour average")
plt.xlabel("Time")
plt.ylabel("Temperature (F)")
plt.title("W06C temperature with rolling average")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 3. Handle gaps honestly

Our dataset has two outage gaps. By default a line plot draws a straight segment
across a gap, which invents data that was never measured and misleads the viewer.
The fix: detect intervals that are much larger than the typical sampling step, and
insert a `NaN` there so matplotlib breaks the line. A broken line is the honest
picture of missing data.

The function below computes the typical spacing between readings, flags any gap
larger than a multiple of it, and inserts a break at each flagged gap.

In [ ]:
def plotWithGaps(frame, valueCol="valueF", gapFactor=3):
    """Plot a time series that breaks the line across missing data."""
    frame = frame.sort_values("timestamp").reset_index(drop=True)
    stepSizes = frame["timestamp"].diff()
    typicalStep = stepSizes.median()
    breakPoints = frame.index[stepSizes > typicalStep * gapFactor].tolist()

    # Insert a NaN row at each detected gap so the line separates there.
    withBreaks = frame.copy()
    for idx in breakPoints:
        breakRow = withBreaks.loc[idx].copy()
        breakRow[valueCol] = float("nan")
        breakRow["timestamp"] = frame.loc[idx, "timestamp"] - typicalStep
        withBreaks = pd.concat([withBreaks, breakRow.to_frame().T], ignore_index=True)
    withBreaks = withBreaks.sort_values("timestamp")

    plt.plot(withBreaks["timestamp"], withBreaks[valueCol],
             color="tab:red", linewidth=1.4)
    print(f"detected {len(breakPoints)} gaps larger than {gapFactor}x the "
          f"typical {typicalStep} step")


plotWithGaps(plotDf)
plt.xlabel("Time")
plt.ylabel("Temperature (F)")
plt.title("W06C temperature, gaps preserved")
plt.grid(True, linestyle="--", alpha=0.5)
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# SageTools packages this whole pattern as plotTemperature, reading from a CSV and
# writing a PNG, with the same raw plus rolling average plus gap handling logic.
if haveSageTools:
    plotDf.to_csv("w06c_plot.csv", index=False)
    from sage.beehive.visualization.plotTemperature import plotTemperature
    plotTemperature("w06c_plot.csv", output="w06c_plot.png", title="W06C temperature")
    print("wrote w06c_plot.png")

## 4. Compare two nodes

To compare sites, overlay their smoothed lines on one axis with a shared scale, so
differences are read off directly. Here we generate a second, cooler node and plot
both. A shared y axis is what makes the comparison fair.

In [ ]:
nodeA = plotDf
nodeB = makeSyntheticBeehive(vsn="W020", days=14, seed=11, sensors=("bme680",))
nodeB["timestamp"] = pd.to_datetime(nodeB["timestamp"])
nodeB["valueF"] = (nodeB["value"] - 3.0) * 9 / 5 + 32   # a cooler site
nodeB = nodeB.sort_values("timestamp")

for frame, label, color in [(nodeA, "W06C", "tab:red"), (nodeB, "W020", "tab:green")]:
    smoothed = frame.set_index("timestamp")["valueF"].rolling("1h", min_periods=1).mean()
    plt.plot(smoothed.index, smoothed.values, color=color, linewidth=1.4, label=label)

plt.xlabel("Time")
plt.ylabel("Temperature (F)")
plt.title("Two node comparison, 1 hour average")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. Put the two nodes on a map

A time series shows how a node changes; a map shows where nodes are and lets you
compare them in space. The cell below places the two nodes and colors each by its
mean temperature, so a warmer site reads warmer on the map. It installs folium
automatically if the kernel does not already have it, using the exact Python
behind this kernel, so the import works without a restart.

New to folium? The supplement notebook `supplement_folium_maps.ipynb` teaches it from scratch and builds up to exactly this map code.

In [ ]:
def ensureFolium():
    """Import folium and branca, installing folium into this kernel if missing.
    Returns (foliumModule, colormapModule) or (None, None) if the install fails."""
    try:
        import folium
        import branca.colormap as colormap
        return folium, colormap
    except ImportError:
        import sys, subprocess, importlib
        print("folium not found, installing it into this kernel...")
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", "folium"],
            capture_output=True, text=True)
        if result.returncode != 0:
            print("folium install failed, skipping the map.")
            return None, None
        importlib.invalidate_caches()
        import folium
        import branca.colormap as colormap
        return folium, colormap


folium, cm = ensureFolium()

if folium is not None:
    from IPython.display import display

    nodeSites = {
        "W06C": (41.87, -87.65, float(nodeA["valueF"].mean())),
        "W020": (41.79, -87.60, float(nodeB["valueF"].mean())),
    }
    lats = [lat for lat, lon, val in nodeSites.values()]
    lons = [lon for lat, lon, val in nodeSites.values()]
    fmap = folium.Map(location=[sum(lats) / len(lats), sum(lons) / len(lons)],
                      zoom_start=11, tiles="OpenStreetMap")

    means = [val for lat, lon, val in nodeSites.values()]
    scale = cm.LinearColormap(["blue", "green", "yellow", "red"],
                              vmin=min(means), vmax=max(means))
    scale.caption = "node mean temperature (F)"
    fmap.add_child(scale)

    for vsn, (lat, lon, val) in nodeSites.items():
        folium.CircleMarker(
            location=[lat, lon], radius=9, color=scale(val), weight=3,
            fill=True, fill_color=scale(val), fill_opacity=0.85,
            popup=f"{vsn}: {val:.1f} F").add_to(fmap)

    display(fmap)
else:
    print("folium unavailable, skipping the map.")

## 6. Exercises

1. Change the rolling window to `6h`, then `1d`. What detail do you lose at each
   step, and what longer trend becomes clearer?
2. In `plotWithGaps`, try `gapFactor=10`. Which gaps stop being detected, and what
   is the danger of setting the factor too high?
3. Add a shaded band of plus and minus one rolling standard deviation around the
   average. Does the band widen during the day or at night?
   *Hint: `.rolling("1h").std()` gives the band half width; use `plt.fill_between`.*
4. Plot the difference between the two nodes (`W06C` minus `W020`) as its own
   line. When is the gap between sites largest? State one hypothesis for why.
5. On the map, change one node's coordinates so the two nodes sit far apart,
   then re-run. How does folium adjust the view to fit both markers?

## Further reading

- matplotlib, the plotting library: https://matplotlib.org/stable/
- matplotlib date handling for time axes: https://matplotlib.org/stable/api/dates_api.html
- pandas `rolling` for moving averages: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rolling.html
- folium, the mapping library used above: https://python-visualization.github.io/folium/
- OpenStreetMap, the map tiles folium uses: https://www.openstreetmap.org/